In [ ]:
import sys
sys.path.insert(0,'..')
from vigipy import *
import duckdb
import pandas as pd
import numpy as np

In [ ]:
def compute_contingency_duckdb(
    parquet_path,
    product_label="drug_name",
    ae_label="reaction_pt",
    margin_threshold=1,
    filter_query=None,
):
    """
    Crea una contingency table da un file parquet usando DuckDB,
    compatibile con vigipy (bcpnn e altri metodi di disproportionality analysis).

    Args:
        parquet_path (str): Path al file .parquet
        product_label (str): Colonna con il nome del farmaco
        ae_label (str): Colonna con l'adverse event
        margin_threshold (int): Numero minimo di eventi per tenere un farmaco/AE
        filter_query (str): Filtro SQL opzionale, es. "receive_year = 2022"

    Returns:
        pd.DataFrame: Contingency table (farmaci x AE) pronta per convert() di vigipy
    """
    con = duckdb.connect()

    where_clause = f"""
        WHERE {product_label} IS NOT NULL 
        AND {ae_label} IS NOT NULL
    """
    if filter_query:
        where_clause += f" AND ({filter_query})"

    query = f"""
        SELECT 
            {product_label} AS name,
            {ae_label}      AS AE,
            COUNT(*)        AS count
        FROM read_parquet('{parquet_path}')
        {where_clause}
        GROUP BY {product_label}, {ae_label}
    """

    drug_adverse = con.execute(query).fetchdf()

    # Forza tipi numpy standard (evita problemi PyArrow con vigipy)
    drug_adverse = pd.DataFrame({
        "name":  np.array(drug_adverse["name"], dtype=str),
        "AE":    np.array(drug_adverse["AE"], dtype=str),
        "count": np.array(drug_adverse["count"], dtype=np.int64),
    })

    # Crea il pivot (contingency table)
    data_cont = pd.pivot_table(
        drug_adverse,
        values="count",
        index="name",
        columns="AE",
        aggfunc="sum",
        fill_value=0,
    ).astype(float)

    data_cont.index = pd.Index(data_cont.index.astype(str).tolist())
    data_cont.columns = pd.Index(data_cont.columns.astype(str).tolist())

    # Applica margin_threshold
    row_mask = np.sum(data_cont.values, axis=1) < margin_threshold
    col_mask = np.sum(data_cont.values, axis=0) < margin_threshold
    data_cont = data_cont.drop(data_cont.index[row_mask])
    data_cont = data_cont.drop(data_cont.columns[col_mask], axis=1)

    con.close()
    return data_cont